# Moodle survey analysis

In [2]:
# import dependencies
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

## Aggregated numbers

In [10]:
# import courseID - courseName - courseCategory for looping
base_table_path = "C:/Users/rpr/OneDrive - Stifterverband/Desktop/Zwischenspeicher/KIC-course completion rate.csv"
base_df = pd.read_csv(base_table_path, sep = ';', encoding="cp1252")

base_df.columns = ['courseID', 'Course', 'Course Category']
print(base_df.info())
base_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   courseID         97 non-null     int64
 1   Course           97 non-null     str  
 2   Course Category  97 non-null     str  
dtypes: int64(1), str(2)
memory usage: 2.4 KB
None


,courseID,Course,Course Category
0,106,Einführung in die KI,Über KI
1,58,AICE your exams – Generative KI als Copilot im...,Studieren
2,99,KI für Alle 1: Einführung in die Künstliche In...,Über KI
3,313,EU AI Act Essentials,Studieren
4,197,KIÖV - KI in öffentlichen Verwaltungen,Studieren


### Import data

In [44]:
# import pre and post surveys --> stack rows together

agg_pre_list = []
agg_post_list = []

#pre_DE = "Umfrage_zu_Kursbeginn.csv"
#pre_EN = "Start_of_course_Questionnaire.csv"

#post_DE = "Umfrage_zum_Kursende.csv"
#post_EN = "End_of_course_Questionnaire.csv"

survey_base_path = "C:/Users/rpr/OneDrive - Stifterverband/Future Skills & KI-Daten - Dokumente/Rio Task Folder/Downloaded Moodle Questionnaire Data/"

for courseID in base_df['courseID']:

    # post survey
    file_path_post_de = os.path.join(survey_base_path, f"Jun26 - {courseID}.csv")
    
    post_path = file_path_post_de if os.path.exists(file_path_post_de) else (file_path_post_en if os.path.exists(file_path_post_en) else None)
    
    if post_path is None:
        print(f"Warning: Post survey file not found for courseID = {courseID}")
    else:
        try:
            post_survey_df = pd.read_csv(post_path, sep = ',')
            post_survey_df['courseID'] = courseID
            
            agg_post_list.append(post_survey_df)
        
        except Exception as e:
            print(f"Warning: Falid reading {post_path} for courseID = {courseID} : {e}")

agg_pre_df = pd.concat(agg_pre_list, ignore_index = True) if agg_pre_list else pd.DataFrame()
agg_post_df = pd.concat(agg_post_list, ignore_index = True) if agg_post_list else pd.DataFrame()


print(agg_post_df)

       Antworten           Abgegeben: Institution  Abteilung  \
0        47245.0  01.10.2025 16:09:18         NaN        NaN   
1       108000.0  24.04.2026 19:34:40         NaN        NaN   
2        52124.0  24.10.2025 21:35:45         NaN        NaN   
3        47209.0  01.10.2025 13:52:16         NaN        NaN   
4        70180.0  26.12.2025 23:05:32         NaN        NaN   
...          ...                  ...         ...        ...   
21064   135930.0  07.07.2026 20:55:12         NaN        NaN   
21065   125237.0  27.06.2026 16:32:22         NaN        NaN   
21066   134990.0  03.07.2026 12:05:23         NaN        NaN   
21067   134024.0  30.06.2026 13:53:49         NaN        NaN   
21068   134984.0  03.07.2026 11:49:14         NaN        NaN   

                                                   Kurs  Gruppe  Nutzer-ID  \
0                                  Einführung in die KI     NaN     1041.0   
1                                  Einführung in die KI     NaN     1893.0 

In [45]:
#print(agg_pre_df.shape)
print(agg_post_df.shape)

# drop unnecessary columns
#agg_pre_df = agg_pre_df.drop(['Response', 'Submitted on:', 'Institution', 'Department', 'Group', 'ID', 'Full name', 'Username'], axis = 1)
agg_post_df = agg_post_df.drop(['Response', 'Submitted on:', 'Institution', 'Department', 'Group', 'ID', 'Full name', 'Username'], axis = 1)

# drop unnecessary columns
#agg_pre_df = agg_pre_df.drop(['Antworten', 'Abgegeben:', 'Abteilung', 'Gruppe', 'Nutzer-ID', 'Vollständiger Name', 'Anmeldename'], axis = 1)
agg_post_df = agg_post_df.drop(['Antworten', 'Abgegeben:', 'Abteilung', 'Gruppe', 'Nutzer-ID', 'Vollständiger Name', 'Anmeldename'], axis = 1)



(21069, 249)


In [18]:
print(agg_pre_df.info())
print(agg_pre_df.shape)
print(agg_pre_df.isnull().sum().shape)

print(agg_post_df.info())
print(agg_post_df.shape)
print(agg_post_df.isnull().sum().shape)

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
None
(0, 0)
(0,)
<class 'pandas.DataFrame'>
RangeIndex: 21069 entries, 0 to 21068
Columns: 234 entries, Kurs to Course
dtypes: float64(211), int64(1), str(22)
memory usage: 37.6 MB
None
(21069, 234)
(234,)


In [14]:
print(len(pd.unique(agg_pre_df.columns)))
print(len(pd.unique(agg_post_df.columns)))

133
194


### Preprocessing

In [20]:
# list cols for standarisation

pre_en_cols = ['Course', \
               'Q01_Question 1->I’m interested in the course topic.', \
               'Q01_Question 1->I want to earn a certificate (e.g. Record of Participation / Achievement).', \
               'Q01_Question 1->It’s part of my university studies (e.g. a required or elective course).', \
               'Q01_Question 1->I want to deepen my knowledge for professional development.', \
               'Q01_Question 1->I’m pursuing a higher professional qualification or career change.', \
               'Q01_Question 1->I already have prior knowledge and am looking to refresh or expand it.', \
               'Q01_Question 1->I want to use the course materials in my own teaching.', \
               'Q01_Question 1->Other', 'Q02_Question 2->Foundations of AI', \
               'Q02_Question 2->Machine Learning', \
               'Q02_Question 2->Natural Language Processing / Large Language Models (NLP / LLMs)', \
               'Q02_Question 2->Data Science', 'Q02_Question 2->Data Literacy', \
               'Q02_Question 2->Robotics', \
               'Q02_Question 2->Human-AI Interaction (e.g. Human-Computer Interaction)', \
               'Q02_Question 2->Internet of Things (IoT) & Industry 4.0', \
               'Q02_Question 2->Other', 'Q03_Question 3->AI in Healthcare', \
               'Q03_Question 3->AI in Education', \
               'Q03_Question 3->AI in Public Administration', \
               'Q03_Question 3->AI for Business & Management', \
               'Q03_Question 3->Ethical and Societal Questions Related to AI', \
               'Q03_Question 3->Other', 'Q04_Question 4->Male', 'Q04_Question 4->Female', 'Q04_Question 4->Diverse', \
               'Q04_Question 4->I’d rather not answer', \
               'Q05_Question 5->Doctorate / PhD', \
               'Q05_Question 5->Master’s degree or equivalent (e.g. Diplom, Magister, postgraduate university degree)', \
               'Q05_Question 5->Bachelor’s degree or equivalent (e.g. B.Sc., B.A., vocational university degree)', \
               'Q05_Question 5->Completed vocational or professional training (e.g. apprenticeship, business school, technical college)', \
               'Q05_Question 5->Upper secondary school certificate (e.g. Abitur, A Levels, Mittlere Reife, GCSEs)', \
               'Q05_Question 5->Lower secondary school certificate (e.g. Realschule, Hauptschule, basic secondary education)', \
               'Q05_Question 5->No formal school-leaving certificate or training', \
               'Q05_Question 5->I’d rather not answer', \
               'Q06_Question 6->Manufacturing, Construction, or Engineering', \
               'Q06_Question 6->Trade, Transport, or Logistics', \
               'Q06_Question 6->Information Technology or Media', \
               'Q06_Question 6->Finance, Insurance, or Real Estate', \
               'Q06_Question 6->Healthcare or Social Services', \
               'Q06_Question 6->Education, Research, or Childcare', \
               'Q06_Question 6->Public Administration or Government', \
               'Q06_Question 6->Hospitality or Tourism', \
               'Q06_Question 6->Other Services', \
               'Q06_Question 6->Other / None of the above', \
               'Q06_Question 6->I’d rather not answer']

post_en_cols = ['Course', \
                'Q01_Question 1->Very good', \
                'Q01_Question 1->Good', 'Q01_Question 1->Average', \
                'Q01_Question 1->Bad', 'Q01_Question 1->Very bad', \
                'Q02_Question 2->Yes', 'Q02_Question 2->No', \
                'Q03_Question 3->The content was not accurate or up-to-date', \
                'Q03_Question 3->The explanations or structure were hard to follow (didactics)', \
                'Q03_Question 3->The course design or layout made it difficult to navigate', \
                'Q03_Question 3->The level of the course did not match my expectations', \
                'Q03_Question 3->The course did not meet my learning goals', \
                'Q03_Question 3->The technical quality (videos, platform, interactions) was poor', \
                'Q03_Question 3->Other', 'Q04_Question 4->Completely', \
                'Q04_Question 4->Mostly', 'Q04_Question 4->Partly', \
                'Q04_Question 4->Not really', 'Q04_Question 4->Not at all', \
                'Q05_Question 5->Extensive', 'Q05_Question 5->Somewhat extensive', \
                'Q05_Question 5->Average', 'Q05_Question 5->Somewhat limited', \
                'Q05_Question 5->Limited', 'Q06_Question 6->Extensive', \
                'Q06_Question 6->Somewhat extensive', 'Q06_Question 6->Average', \
                'Q06_Question 6->Somewhat limited', 'Q06_Question 6->Limited', \
                'Q07_Question 7->Short videos or video lectures', \
                'Q07_Question 7->Written explanations or reading materials', \
                'Q07_Question 7->Practice exercises (e.g. programming tasks, quizzes)', \
                'Q07_Question 7->Visual aids (e.g. diagrams, infographics, simulations)', \
                'Q07_Question 7->Final test or exam', \
                'Q07_Question 7->Peer discussion or forum participation', \
                'Q07_Question 7->AI Chatbot (KI-Lernassistent)', \
                'Q07_Question 7->Other', 'Q08_Question 8->Very helpful', \
                'Q08_Question 8->Helpful', 'Q08_Question 8->Neutral', \
                'Q08_Question 8->Not very helpful', \
                'Q08_Question 8->Not helpful at all', \
                'Q08_Question 8->I haven’t used it', \
                'Q09_Question 9->To get explanations of specific concepts', \
                'Q09_Question 9->To summarise course content (e.g. texts, videos, exercises)', \
                'Q09_Question 9->To get content recommendations or navigation help', \
                'Q09_Question 9->To prepare for quizzes or the final test', \
                'Q09_Question 9->To help with programming tasks or exercises', \
                'Q09_Question 9->To ask administrative or organisational questions', \
                'Q09_Question 9->I tried it out, but didn’t use it seriously', \
                'Q09_Question 9->I didn’t use the chatbot', \
                'Q09_Question 9->Other (please specify)', 'Q10_Question 10->Female', \
                'Q10_Question 10->Male', 'Q10_Question 10->Other', \
                'Q10_Question 10->I’d rather not answer', \
                'Q11_Question 11->Doctorate / PhD', \
                'Q11_Question 11->Master’s degree or equivalent (e.g. Diplom, Magister, postgraduate university degree)', \
                'Q11_Question 11->Bachelor’s degree or equivalent (e.g. B.Sc., B.A., vocational university degree)', \
                'Q11_Question 11->Completed vocational or professional training (e.g. apprenticeship, business school, technical college)', \
                'Q11_Question 11->Upper secondary school certificate (e.g. Abitur, A Levels, Mittlere Reife, GCSEs)', \
                'Q11_Question 11->Lower secondary school certificate (e.g. Realschule, Hauptschule, basic secondary education)', \
                'Q11_Question 11->No formal school-leaving certificate or training', \
                'Q11_Question 11->I’d rather not answer', \
                'Q12_Question 12->Manufacturing, Construction, or Engineering', \
                'Q12_Question 12->Trade, Transport, or Logistics', \
                'Q12_Question 12->Information Technology or Media', \
                'Q12_Question 12->Finance, Insurance, or Real Estate', \
                'Q12_Question 12->Healthcare or Social Services', \
                'Q12_Question 12->Education, Research, or Childcare', \
                'Q12_Question 12->Public Administration or Government', \
                'Q12_Question 12->Hospitality or Tourism', \
                'Q12_Question 12->Other Services', \
                'Q12_Question 12->Other / None of the above', \
                'Q12_Question 12->I’d rather not answer']

pre_de_cols = ['Course', \
               'Q01_Question 1->Ich interessiere mich für das Kursthema.', \
               'Q01_Question 1->Ich möchte ein Zertifikat erhalten (zB. Teilnahmebestätigung, Leistungsnachweis).', \
               'Q01_Question 1->Es ist Teil meines Universitätsstudiums (z. B. ein Pflicht- oder Wahlfach).', \
               'Q01_Question 1->Ich möchte mein Wissen zur beruflichen Weiterentwicklung vertiefen.', \
               'Q01_Question 1->Ich strebe eine berufliche Höherqualifizierung oder einen Berufswechsel an.', \
               'Q01_Question 1->Ich verfüge bereits über Vorkenntnisse und möchte diese auffrischen oder erweitern.', \
               'Q01_Question 1->Ich möchte die Kursmaterialien in meinem eigenen Unterricht verwenden.', \
               'Q01_Question 1->Sonstiges', 'Q02_Question 2->Grundlagen der KI', \
               'Q02_Question 2->Maschinelles Lernen', \
               'Q02_Question 2->Natural Language Processing / Large Language Models (NLP / LL.M.)', \
               'Q02_Question 2->Data Science', 'Q02_Question 2->Datenkompetenz', \
               'Q02_Question 2->Robotik', \
               'Q02_Question 2->Mensch-KI-Interaktion (z. B. Mensch-Computer-Interaktion)', \
               'Q02_Question 2->Internet der Dinge (IoT) & Industrie 4.0', \
               'Q02_Question 2->Sonstiges', 'Q03_Question 3->KI im Gesundheitswesen', \
               'Q03_Question 3->KI im Bildungswesen', \
               'Q03_Question 3->KI in der öffentlichen Verwaltung', \
               'Q03_Question 3->KI für Wirtschaft und Management', \
               'Q03_Question 3->Ethische und gesellschaftliche Fragen im Zusammenhang mit KI', \
               'Q03_Question 3->Sonstiges', 'Q04_Question 4->Männlich', \
               'Q04_Question 4->Weiblich', 'Q04_Question 4->Divers', \
               'Q04_Question 4->Ich möchte diese Frage nicht beantworten', \
               'Q05_Question 5->Promotion', \
               'Q05_Question 5->Masterabschluss oder gleichwertiger Abschluss (z. B. Diplom, Magister, postgradualer Hochschulabschluss)', \
               'Q05_Question 5->Bachelorabschluss oder gleichwertiger Abschluss (z. B. B.Sc., B.A., Fachhochschulabschluss)', \
               'Q05_Question 5->Abgeschlossene Berufsausbildung (z. B. Lehre, Handelsschule, Fachoberschule)', \
               'Q05_Question 5->Abitur (z. B. Abitur, Mittlere Reife, GCSE)', \
               'Q05_Question 5->Hauptschulabschluss (z. B. Realschule, Hauptschule)', \
               'Q05_Question 5->Kein Schulabschluss oder Ausbildung', \
               'Q05_Question 5->Ich möchte diese Frage nicht beantworten', \
               'Q06_Question 6->Fertigung, Bauwesen oder Ingenieurwesen', \
               'Q06_Question 6->Handel, Transport oder Logistik', \
               'Q06_Question 6->Informationstechnologie oder Medien', \
               'Q06_Question 6->Finanzen, Versicherungen oder Immobilien', \
               'Q06_Question 6->Gesundheits- oder Sozialwesen', \
               'Q06_Question 6->Bildung, Forschung oder Kinderbetreuung', \
               'Q06_Question 6->Öffentliche Verwaltung oder Regierung', \
               'Q06_Question 6->Gastgewerbe oder Tourismus', \
               'Q06_Question 6->Sonstige Dienstleistungen', \
               'Q06_Question 6->Sonstige / Keine der oben genannten', \
               'Q06_Question 6->Ich möchte lieber nicht antworten']

post_de_cols = ['Course', \
                'Q01_Question 1->Sehr gut', \
                'Q01_Question 1->Gut', 'Q01_Question 1->Durchschnittlich', \
                'Q01_Question 1->Schlecht', 'Q01_Question 1->Sehr schlecht', \
                'Q02_Question 2->Ja', 'Q02_Question 2->Nein', \
                'Q03_Question 3->Die Inhalte waren nicht korrekt oder aktuell.', \
                'Q03_Question 3->Die Erklärungen oder die Struktur waren schwer verständlich (Didaktik).', \
                'Q03_Question 3->Das Kursdesign oder -layout erschwerte die Navigation.', \
                'Q03_Question 3->Das Kursniveau entsprach nicht meinen Erwartungen.', \
                'Q03_Question 3->Der Kurs entsprach nicht meinen Lernzielen.', \
                'Q03_Question 3->Die technische Qualität (Videos, Plattform, Interaktionen) war mangelhaft.', \
                'Q03_Question 3->Sonstiges', 'Q04_Question 4->Vollständig', \
                'Q04_Question 4->Größtenteils', 'Q04_Question 4->Teilweise', \
                'Q04_Question 4->Nicht wirklich', 'Q04_Question 4->Überhaupt nicht', \
                'Q05_Question 5->Umfangreich', 'Q05_Question 5->Etwas umfangreich', \
                'Q05_Question 5->Durchschnittlich', \
                'Q05_Question 5->Etwas eingeschränkt', 'Q05_Question 5->Eingeschränkt', \
                'Q06_Question 6->Umfangreich', 'Q06_Question 6->Etwas umfangreich', \
                'Q06_Question 6->Durchschnittlich', \
                'Q06_Question 6->Etwas eingeschränkt', 'Q06_Question 6->Eingeschränkt', \
                'Q07_Question 7->Kurze Videos oder Videovorträge', \
                'Q07_Question 7->Schriftliche Erklärungen oder Lesematerial', \
                'Q07_Question 7->Übungen (z. B. Programmieraufgaben, Quizze)', \
                'Q07_Question 7->Visuelle Hilfsmittel (z. B. Diagramme, Infografiken, Simulationen)', \
                'Q07_Question 7->Abschlusstest oder Prüfung', \
                'Q07_Question 7->Diskussion mit anderen Teilnehmern oder Teilnahme am Forum', \
                'Q07_Question 7->KI-Chatbot (KI-Lernassistent)', \
                'Q07_Question 7->Sonstiges', 'Q08_Question 8->Sehr hilfreich', \
                'Q08_Question 8->Hilfreich', 'Q08_Question 8->Neutral', \
                'Q08_Question 8->Nicht sehr hilfreich', \
                'Q08_Question 8->Gar nicht hilfreich', \
                'Q08_Question 8->Ich habe ihn nicht genutzt', \
                'Q09_Question 9->Um Erklärungen zu bestimmten Konzepten zu erhalten', \
                'Q09_Question 9->Um Kursinhalte (z. B. Texte, Videos, Übungen) zusammenzufassen', \
                'Q09_Question 9->Um Inhaltsempfehlungen oder Navigationshilfen zu erhalten', \
                'Q09_Question 9->Um sich auf Quizze oder die Abschlussprüfung vorzubereiten', \
                'Q09_Question 9->Um bei Programmieraufgaben oder Übungen zu helfen', \
                'Q09_Question 9->Um administrative oder organisatorische Fragen zu stellen', \
                'Q09_Question 9->Ich habe es ausprobiert, aber nicht ernsthaft genutzt', \
                'Q09_Question 9->Ich habe den Chatbot nicht genutzt', \
                'Q09_Question 9->Sonstiges (bitte angeben)', \
                'Q10_Question 10->Weiblich', 'Q10_Question 10->Männlich', \
                'Q10_Question 10->Divers', \
                'Q10_Question 10->Ich möchte diese Frage nicht beantworten', \
                'Q11_Question 11->Promotion', \
                'Q11_Question 11->Masterabschluss oder gleichwertiger Abschluss (z. B. Diplom, Magister, postgradualer Hochschulabschluss)', \
                'Q11_Question 11->Bachelorabschluss oder gleichwertiger Abschluss (z. B. B.Sc., B.A., Fachhochschulabschluss)', \
                'Q11_Question 11->Abgeschlossene Berufsausbildung (z. B. Lehre, Handelsschule, Fachoberschule)', \
                'Q11_Question 11->Abitur (z. B. Abitur, Mittlere Reife, GCSE)', \
                'Q11_Question 11->Hauptschulabschluss (z. B. Realschule, Hauptschule)', \
                'Q11_Question 11->Kein Schulabschluss oder Ausbildung', \
                'Q11_Question 11->Ich möchte diese Frage nicht beantworten', \
                'Q12_Question 12->Fertigung, Bauwesen oder Ingenieurwesen', \
                'Q12_Question 12->Handel, Transport oder Logistik', \
                'Q12_Question 12->Informationstechnologie oder Medien', \
                'Q12_Question 12->Finanzen, Versicherungen oder Immobilien', \
                'Q12_Question 12->Gesundheits- oder Sozialwesen', \
                'Q12_Question 12->Bildung, Forschung oder Kinderbetreuung', \
                'Q12_Question 12->Öffentliche Verwaltung oder Regierung', \
                'Q12_Question 12->Gastgewerbe oder Tourismus', \
                'Q12_Question 12->Sonstige Dienstleistungen',  \
                'Q12_Question 12->Sonstige / Keine der oben genannten', \
                'Q12_Question 12->Ich möchte lieber nicht antworten']
            

In [21]:
#  create mapping dictionaries
print(len(pre_de_cols), len(pre_en_cols))
print(len(post_de_cols), len(post_en_cols))

en2de_pre = dict(zip(pre_en_cols, pre_de_cols))

en2de_post = dict(zip(post_en_cols, post_de_cols))

# convert one-to-one dictionary to 
en2de_pre = {k : [v] for k, v in en2de_pre.items()}

en2de_post = {k : [v] for k, v in en2de_post.items()}


47 47
76 76


In [22]:
# mapping misalignment
# pre misalignment
mis_en_pre = ['Course', \
               'Q02_Question 2->Foundations of AI', \
               'Q02_Question 2->Machine Learning', \
               'Q02_Question 2->Natural Language Processing / Large Language Models (NLP / LLMs)', \
               'Q02_Question 2->Data Science', 'Q02_Question 2->Data Literacy', \
               'Q02_Question 2->Robotics', \
               'Q02_Question 2->Human-AI Interaction (e.g. Human-Computer Interaction)', \
               'Q02_Question 2->Internet of Things (IoT) & Industry 4.0', \
               'Q02_Question 2->Other', 'Q03_Question 3->AI in Healthcare', \
               'Q03_Question 3->AI in Education', \
               'Q03_Question 3->AI in Public Administration', \
               'Q03_Question 3->AI for Business & Management', \
               'Q03_Question 3->Ethical and Societal Questions Related to AI', \
               'Q03_Question 3->Other', 'Q04_Question 4->Male', 'Q04_Question 4->Female', 'Q04_Question 4->Diverse', \
               'Q04_Question 4->I’d rather not answer', \
               'Q05_Question 5->Doctorate / PhD', \
               'Q05_Question 5->Master’s degree or equivalent (e.g. Diplom, Magister, postgraduate university degree)', \
               'Q05_Question 5->Bachelor’s degree or equivalent (e.g. B.Sc., B.A., vocational university degree)', \
               'Q05_Question 5->Completed vocational or professional training (e.g. apprenticeship, business school, technical college)', \
               'Q05_Question 5->Upper secondary school certificate (e.g. Abitur, A Levels, Mittlere Reife, GCSEs)', \
               'Q05_Question 5->Lower secondary school certificate (e.g. Realschule, Hauptschule, basic secondary education)', \
               'Q05_Question 5->No formal school-leaving certificate or training', \
               'Q05_Question 5->I’d rather not answer', \
               'Q06_Question 6->Manufacturing, Construction, or Engineering', \
               'Q06_Question 6->Trade, Transport, or Logistics', \
               'Q06_Question 6->Information Technology or Media', \
               'Q06_Question 6->Finance, Insurance, or Real Estate', \
               'Q06_Question 6->Healthcare or Social Services', \
               'Q06_Question 6->Education, Research, or Childcare', \
               'Q06_Question 6->Public Administration or Government', \
               'Q06_Question 6->Hospitality or Tourism', \
               'Q06_Question 6->Other Services', \
               'Q06_Question 6->Other / None of the above', \
               'Q06_Question 6->I’d rather not answer']

mis_de_pre = ['Kurs', 'Q03_Question 2->Grundlagen der KI', \
              'Q03_Question 2->Maschinelles Lernen', \
              'Q03_Question 2->Natural Language Processing / Large Language Models (NLP / LL.M.)', \
              'Q03_Question 2->Data Science', 'Q03_Question 2->Datenkompetenz', \
              'Q03_Question 2->Robotik', \
              'Q03_Question 2->Mensch-KI-Interaktion (z. B. Mensch-Computer-Interaktion)', \
              'Q03_Question 2->Internet der Dinge (IoT) & Industrie 4.0', \
              'Q03_Question 2->Sonstiges', 'Q04_Question 3->KI im Gesundheitswesen', \
              'Q04_Question 3->KI im Bildungswesen', \
              'Q04_Question 3->KI in der öffentlichen Verwaltung', \
              'Q04_Question 3->KI für Wirtschaft und Management', \
              'Q04_Question 3->Ethische und gesellschaftliche Fragen im Zusammenhang mit KI', \
              'Q04_Question 3->Sonstiges', 'Q05_Question 4->Männlich', \
              'Q05_Question 4->Weiblich', 'Q05_Question 4->Divers', \
              'Q05_Question 4->Ich möchte diese Frage nicht beantworten', \
              'Q06_Question 5->Promotion', \
              'Q06_Question 5->Masterabschluss oder gleichwertiger Abschluss (z. B. Diplom, Magister, postgradualer Hochschulabschluss)', \
              'Q06_Question 5->Bachelorabschluss oder gleichwertiger Abschluss (z. B. B.Sc., B.A., Fachhochschulabschluss)', \
              'Q06_Question 5->Abgeschlossene Berufsausbildung (z. B. Lehre, Handelsschule, Fachoberschule)', \
              'Q06_Question 5->Abitur (z. B. Abitur, Mittlere Reife, GCSE)', \
              'Q06_Question 5->Hauptschulabschluss (z. B. Realschule, Hauptschule)', \
              'Q06_Question 5->Kein Schulabschluss oder Ausbildung', \
              'Q06_Question 5->Ich möchte diese Frage nicht beantworten', \
              'Q07_Question 6->Fertigung, Bauwesen oder Ingenieurwesen', \
              'Q07_Question 6->Handel, Transport oder Logistik', \
              'Q07_Question 6->Informationstechnologie oder Medien', \
              'Q07_Question 6->Finanzen, Versicherungen oder Immobilien', \
              'Q07_Question 6->Gesundheits- oder Sozialwesen', \
              'Q07_Question 6->Bildung, Forschung oder Kinderbetreuung', \
              'Q07_Question 6->Öffentliche Verwaltung oder Regierung', \
              'Q07_Question 6->Gastgewerbe oder Tourismus', \
              'Q07_Question 6->Sonstige Dienstleistungen', \
              'Q07_Question 6->Sonstige / Keine der oben genannten', \
              'Q07_Question 6->Ich möchte lieber nicht antworten']

# post misalignment
mis_en_post = ['Course', \
              'Q08_Question 8->Very helpful', \
              'Q08_Question 8->Helpful', 'Q08_Question 8->Neutral', \
              'Q08_Question 8->Not very helpful', \
              'Q08_Question 8->Not helpful at all', \
              'Q08_Question 8->I haven’t used it', \
              'Q09_Question 9->To get explanations of specific concepts', \
              'Q09_Question 9->To summarise course content (e.g. texts, videos, exercises)', \
              'Q09_Question 9->To get content recommendations or navigation help', \
              'Q09_Question 9->To prepare for quizzes or the final test', \
              'Q09_Question 9->To help with programming tasks or exercises', \
              'Q09_Question 9->To ask administrative or organisational questions', \
              'Q09_Question 9->I tried it out, but didn’t use it seriously', \
              'Q09_Question 9->I didn’t use the chatbot', \
              'Q09_Question 9->Other (please specify)', 'Q10_Question 10->Female', \
              'Q10_Question 10->Male', 'Q10_Question 10->Other', \
              'Q10_Question 10->I’d rather not answer', \
              'Q11_Question 11->Doctorate / PhD', \
              'Q11_Question 11->Master’s degree or equivalent (e.g. Diplom, Magister, postgraduate university degree)', \
              'Q11_Question 11->Bachelor’s degree or equivalent (e.g. B.Sc., B.A., vocational university degree)', \
              'Q11_Question 11->Completed vocational or professional training (e.g. apprenticeship, business school, technical college)', \
              'Q11_Question 11->Upper secondary school certificate (e.g. Abitur, A Levels, Mittlere Reife, GCSEs)', \
              'Q11_Question 11->Lower secondary school certificate (e.g. Realschule, Hauptschule, basic secondary education)', \
              'Q11_Question 11->No formal school-leaving certificate or training', \
              'Q11_Question 11->I’d rather not answer', \
              'Q12_Question 12->Manufacturing, Construction, or Engineering', \
              'Q12_Question 12->Trade, Transport, or Logistics', \
              'Q12_Question 12->Information Technology or Media', \
              'Q12_Question 12->Finance, Insurance, or Real Estate', \
              'Q12_Question 12->Healthcare or Social Services', \
              'Q12_Question 12->Education, Research, or Childcare', \
              'Q12_Question 12->Public Administration or Government', \
              'Q12_Question 12->Hospitality or Tourism', \
              'Q12_Question 12->Other Services', \
              'Q12_Question 12->Other / None of the above', \
              'Q12_Question 12->I’d rather not answer']

mis_de_post = ['Kurs', 'Q10_Question 8->Sehr hilfreich', 'Q10_Question 8->Hilfreich', \
               'Q10_Question 8->Neutral', 'Q10_Question 8->Nicht sehr hilfreich', \
               'Q10_Question 8->Gar nicht hilfreich', \
               'Q10_Question 8->Ich habe ihn nicht genutzt', \
               'Q11_Question 9->Um Erklärungen zu bestimmten Konzepten zu erhalten', \
               'Q11_Question 9->Um Kursinhalte (z. B. Texte, Videos, Übungen) zusammenzufassen', \
               'Q11_Question 9->Um Inhaltsempfehlungen oder Navigationshilfen zu erhalten', \
               'Q11_Question 9->Um sich auf Quizze oder die Abschlussprüfung vorzubereiten', \
               'Q11_Question 9->Um bei Programmieraufgaben oder Übungen zu helfen', \
               'Q11_Question 9->Um administrative oder organisatorische Fragen zu stellen', \
               'Q11_Question 9->Ich habe es ausprobiert, aber nicht ernsthaft genutzt', \
               'Q11_Question 9->Ich habe den Chatbot nicht genutzt', \
               'Q11_Question 9->Sonstiges (bitte angeben)', \
               'Q12_Question 10->Weiblich', 'Q12_Question 10->Männlich', \
               'Q12_Question 10->Divers',\
               'Q12_Question 10->Ich möchte diese Frage nicht beantworten', \
               'Q13_Question 11->Promotion', \
               'Q13_Question 11->Masterabschluss oder gleichwertiger Abschluss (z. B. Diplom, Magister, postgradualer Hochschulabschluss)', \
               'Q13_Question 11->Bachelorabschluss oder gleichwertiger Abschluss (z. B. B.Sc., B.A., Fachhochschulabschluss)', \
               'Q13_Question 11->Abgeschlossene Berufsausbildung (z. B. Lehre, Handelsschule, Fachoberschule)', \
               'Q13_Question 11->Abitur (z. B. Abitur, Mittlere Reife, GCSE)', \
               'Q13_Question 11->Hauptschulabschluss (z. B. Realschule, Hauptschule)', \
               'Q13_Question 11->Kein Schulabschluss oder Ausbildung', \
               'Q13_Question 11->Ich möchte diese Frage nicht beantworten', \
               'Q14_Question 12->Fertigung, Bauwesen oder Ingenieurwesen', \
               'Q14_Question 12->Handel, Transport oder Logistik', \
               'Q14_Question 12->Informationstechnologie oder Medien', \
               'Q14_Question 12->Finanzen, Versicherungen oder Immobilien', \
               'Q14_Question 12->Gesundheits- oder Sozialwesen', \
               'Q14_Question 12->Bildung, Forschung oder Kinderbetreuung', \
               'Q14_Question 12->Öffentliche Verwaltung oder Regierung', \
               'Q14_Question 12->Gastgewerbe oder Tourismus', \
               'Q14_Question 12->Sonstige Dienstleistungen', \
               'Q14_Question 12->Sonstige / Keine der oben genannten', \
               'Q14_Question 12->Ich möchte lieber nicht antworten']



# create match
mis_pre = dict(zip(mis_en_pre, mis_de_pre))
mis_post = dict(zip(mis_en_post, mis_de_post))


In [23]:
# add values to the existing list

for k, v in mis_pre.items():
    en2de_pre[k].append(v)
        
for k, v in mis_post.items():
    en2de_post[k].append(v)


In [24]:
def standard_df_builder(og_df : pd.DataFrame, map_dic : dict, canonical_cols : list, meta_first :  list | None = None) -> pd.DataFrame:
    meta_first = meta_first or []
    meta_first = [c for c in meta_first if c in og_df.columns]
    
    out = pd.DataFrame(index = og_df.index)
    
    for en_col in canonical_cols:
        candidates = []
        
        if en_col in og_df.columns:
            candidates.append(og_df[en_col])
        
        for de_col in map_dic.get(en_col, []):
            if de_col in og_df.columns:
                candidates.append(og_df[de_col])
                
        if not candidates:
            out[en_col] = pd.NA
        
        else:
            out[en_col] = pd.concat(candidates, axis = 1).bfill(axis = 1).iloc[: , 0]
            
    canonical_set = set(canonical_cols)
    de_cols4map = set()
    for val_list in map_dic.values():
        de_cols4map.update(val_list)
    
    extras = []
    
    for col in og_df.columns:
        if col in meta_first:
            continue
        if col in canonical_set:
            continue
        if col in de_cols4map:
            continue
        extras.append(col)
        
    parts = []
    
    if meta_first:
        parts.append(og_df[meta_first])
    parts.append(out)
    if extras:
        parts.append(og_df[extras])
    
    return pd.concat(parts, axis = 1)
     

In [25]:
# another approach: create standardised dataframe and grab colunms to it, extra columns will be added to the standard dataframe
# for pre survey
agg_pre = standard_df_builder(agg_pre_df, en2de_pre, pre_en_cols, meta_first = ['courseID'])

agg_pre.shape

(0, 47)

In [26]:
agg_pre.columns

Index(['Course', 'Q01_Question 1->I’m interested in the course topic.',
       'Q01_Question 1->I want to earn a certificate (e.g. Record of Participation / Achievement).',
       'Q01_Question 1->It’s part of my university studies (e.g. a required or elective course).',
       'Q01_Question 1->I want to deepen my knowledge for professional development.',
       'Q01_Question 1->I’m pursuing a higher professional qualification or career change.',
       'Q01_Question 1->I already have prior knowledge and am looking to refresh or expand it.',
       'Q01_Question 1->I want to use the course materials in my own teaching.',
       'Q01_Question 1->Other', 'Q02_Question 2->Foundations of AI',
       'Q02_Question 2->Machine Learning',
       'Q02_Question 2->Natural Language Processing / Large Language Models (NLP / LLMs)',
       'Q02_Question 2->Data Science', 'Q02_Question 2->Data Literacy',
       'Q02_Question 2->Robotics',
       'Q02_Question 2->Human-AI Interaction (e.g. Human-Comp

In [27]:
# for post survey

agg_post = standard_df_builder(agg_post_df, en2de_post, post_en_cols, meta_first = ['courseID'])

agg_post.shape

(21069, 121)

In [28]:
agg_post.columns

Index(['courseID', 'Course', 'Q01_Question 1->Very good',
       'Q01_Question 1->Good', 'Q01_Question 1->Average',
       'Q01_Question 1->Bad', 'Q01_Question 1->Very bad',
       'Q02_Question 2->Yes', 'Q02_Question 2->No',
       'Q03_Question 3->The content was not accurate or up-to-date',
       ...
       'Q11_Frage 11', 'Q12_Frage 12', 'Q13_Frage 13', 'Q14_Frage 14',
       'Q15_Frage 15', 'Q16_Frage 16', 'Q17_Frage 17', 'Q18_Frage 18',
       'Q08_Gut gefallen', 'Q09_Anregungen'],
      dtype='str', length=121)

In [29]:
# check nan
agg_pre.isnull().sum()
agg_post.isnull().sum()

courseID                         0
Course                           0
Q01_Question 1->Very good      310
Q01_Question 1->Good           310
Q01_Question 1->Average        310
                             ...  
Q16_Frage 16                 20759
Q17_Frage 17                 20759
Q18_Frage 18                 20759
Q08_Gut gefallen             20990
Q09_Anregungen               21013
Length: 121, dtype: int64

### Calculating mean of Q1 and Q4

In [62]:
# data for Q1 and Q4 per courseID

#Q1: Columns with corresponding scores
q1_cols = {
    'Q01_Question 1->Very good': 1,
    'Q01_Question 1->Good': 2,
    'Q01_Question 1->Average': 3,
    'Q01_Question 1->Bad': 4,
    'Q01_Question 1->Very bad': 5
}

#Q1: Summing the different values:
agg_post['mean_Q1'] = (
    agg_post[list(q1_cols.keys())]
    .mul(pd.Series(q1_cols))
    .sum(axis=1)
)

#Q1: calculating the mean of the values and round to 2 integers after comma
Q1_course_mean = (
    agg_post
    .groupby('courseID')['mean_Q1']
    .mean()
    .round(2)
    .reset_index()
)

#Q4: Columns with corresponding scores
q4_cols = {
    'Q04_Question 4->Completely': 1,
    'Q04_Question 4->Mostly': 2,
    'Q04_Question 4->Partly': 3,
    'Q04_Question 4->Not really': 4,
    'Q04_Question 4->Not at all': 5
}

#Q4: Summing the different values:
agg_post['mean_Q4'] = (
    agg_post[list(q4_cols.keys())]
    .mul(pd.Series(q4_cols))
    .sum(axis=1)
)

#Q4: calculating the mean of the values and round to 2 integers after comma
Q4_course_mean = (
    agg_post
    .groupby('courseID')['mean_Q4']
    .mean()
    .round(2)
    .reset_index()
)


#merging Q1 and Q4 into one table
course_means = (
    Q1_course_mean
    .merge(Q4_course_mean, on="courseID")
)

#exporting csv of merged df to wd
course_means.to_csv("course_question_means.csv", index=False)

   courseID                Course  Q01_Question 1->Very good  \
0       106  Einführung in die KI                        0.0   
1       106  Einführung in die KI                        1.0   
2       106  Einführung in die KI                        1.0   
3       106  Einführung in die KI                        0.0   
4       106  Einführung in die KI                        0.0   

   Q01_Question 1->Good  Q01_Question 1->Average  Q01_Question 1->Bad  \
0                   1.0                      0.0                  0.0   
1                   0.0                      0.0                  0.0   
2                   0.0                      0.0                  0.0   
3                   1.0                      0.0                  0.0   
4                   1.0                      0.0                  0.0   

   Q01_Question 1->Very bad  Q02_Question 2->Yes  Q02_Question 2->No  \
0                       0.0                  1.0                 0.0   
1                       0.0     

C:\Users\rpr\AppData\Local\Temp\ipykernel_19676\29693726.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_post['mean_Q1'] = (
C:\Users\rpr\AppData\Local\Temp\ipykernel_19676\29693726.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_post['mean_Q4'] = (
